# The backbone fuses the views in its middle layers

Section 5.3 (Fig. 6c): the readout of S-Mind reads a learned mix of all layers. Here a fresh readout is fit on the
states of $\mathbf{z}_0$ at **one layer at a time**, on the checkpoint after geometry pre-training, and scored on the
416 held-out ScanNet image sets; a readout fit on all-zero states gives the floor, and one fit on the learned mix the
ceiling.

**Needs:** the geometry supervision (`scripts/data/geometry.sh`: `data/smind/fusion`); one GPU to cache the states (about 15 minutes), then the
34 fits, which run on a GPU or, faster in parallel, on CPU cores (`--device cpu`; about 20 minutes).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import smind_nb as nb
nb.setup()

SIZE = "4b"   # or "9b"
GPU = "0"

In [ ]:
nb.need("data/smind/fusion/maps_scannet_dev.parquet")
ckpt, out = nb.checkpoint(SIZE, "fusion"), nb.RUNS / f"{SIZE}-fusion/layer_readout"
nb.run(["python", "analysis/method/layer_readout.py", "cache", "--ckpt", ckpt, "--cache", out / "cache"],
       out / "cache.log", done=out / "cache/maps.pt", gpu=GPU)
LAYERS = len(list((out / "cache").glob("layer_*.pt")))
import concurrent.futures, os
parts = [["zero"], ["mix"]] + [[str(l)] for l in range(LAYERS)]
def fit(part):
    nb.run(["python", "analysis/method/layer_readout.py", "probe", "--ckpt", ckpt, "--cache", out / "cache",
            "--layers", *part, "--device", "cpu", "--out", out / f"probe_{part[0]}.json"],
           out / f"probe_{part[0]}.log", done=out / f"probe_{part[0]}.json")
os.environ["OMP_NUM_THREADS"] = "4"
with concurrent.futures.ThreadPoolExecutor(max_workers=min(len(parts), max(1, (os.cpu_count() or 8) // 8))) as pool:
    list(pool.map(fit, parts))

In [ ]:
probe = {}
for f in sorted(out.glob("probe_*.json")):
    probe.update(nb.read_json(f)["layers"])
fused = lambda k: 100 * probe[k]["scannet"]["point_far_vk"]
single = lambda k: 100 * probe[k]["scannet"]["point_far_v1"]
layers = sorted(int(k) for k in probe if k.isdigit())
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(layers, [single(str(l)) for l in layers], "o-", ms=3, color="0.6", label="single-view points")
ax.plot(layers, [fused(str(l)) for l in layers], "o-", ms=3, color="#7A55E6", label="fused points")
ax.axhline(fused("mix"), ls="--", color="#7A55E6", lw=0.8); ax.axhline(fused("zero"), ls=":", color="0.3", lw=0.8)
ax.set_xlabel("layer"); ax.set_ylabel("% of ScanNet dev points"); ax.legend(frameon=False)
plt.tight_layout()
rows = {"all-zero states (fused)": fused("zero"), "learned mix (fused)": fused("mix"),
        "layers 0-8 (fused, min)": min(fused(str(l)) for l in range(9)),
        "layers 0-8 (fused, max)": max(fused(str(l)) for l in range(9)),
        "layers 19+ (fused, min)": min(fused(str(l)) for l in layers if l >= 19),
        "layers 19+ (fused, max)": max(fused(str(l)) for l in layers if l >= 19)}
nb.compare(rows, dict(zip(rows, [9.21, 63.99, 7.64, 11.12, 57.33, 63.54])) if SIZE == "4b" else None)

The fused points stay at the floor through the early layers, rise through the middle layers, and from about layer 19 on
every single layer holds nearly as much as the learned mix.